In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IcebergLakehouse") \
    .master("spark://spark-master:7077") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
        "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
    ])) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hive_prod", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hive_prod.type", "hive") \
    .config("spark.sql.catalog.hive_prod.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.hive_prod.warehouse", "s3a://warehouse/") \
    .config("spark.sql.defaultCatalog", "hive_prod") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

25/09/20 06:03:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
spark.sql("SHOW NAMESPACES").show()  # Should list 'default', 'demo', etc.

+---------+
|namespace|
+---------+
|  default|
+---------+



In [3]:
from minio import Minio
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)

In [4]:
bucket = 'warehouse'
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

In [5]:
spark.sql("CREATE DATABASE IF NOT EXISTS demo")

DataFrame[]

In [7]:
spark.sql("CREATE DATABASE IF NOT EXISTS hive_prod.demo")

DataFrame[]

In [8]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS hive_prod.demo.nyc_taxis (
        vendor_id BIGINT,
        trip_id STRING,
        trip_distance DOUBLE,
        fare_amount DOUBLE,
        store_and_fwd_flag STRING,
        trip_start_timestamp TIMESTAMP
    ) USING iceberg
    PARTITIONED BY (days(trip_start_timestamp))
    TBLPROPERTIES ('format-version' = '2')
    LOCATION 's3a://warehouse/demo/nyc_taxis'
""")

25/09/20 06:05:22 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[]

In [9]:
spark.sql("SHOW TABLES IN hive_prod.demo").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     demo|nyc_taxis|      false|
+---------+---------+-----------+



In [10]:
from pyspark.sql.functions import lit, to_timestamp

In [11]:
data = [
    (1, "trip1", 2.5, 10.0, "N", "2025-09-20 10:00:00"),
    (2, "trip2", 3.7, 15.5, "Y", "2025-09-20 12:30:00"),
    (1, "trip3", 1.2, 7.0, "N", "2025-09-21 08:15:00")
]
df = spark.createDataFrame(data, ["vendor_id", "trip_id", "trip_distance", "fare_amount", "store_and_fwd_flag", "trip_start_timestamp"]) \
    .withColumn("trip_start_timestamp", to_timestamp("trip_start_timestamp"))

In [12]:
df.write.mode("append").saveAsTable("hive_prod.demo.nyc_taxis")

In [13]:
spark.sql("SELECT * FROM hive_prod.demo.nyc_taxis").show()

+---------+-------+-------------+-----------+------------------+--------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|trip_start_timestamp|
+---------+-------+-------------+-----------+------------------+--------------------+
|        1|  trip3|          1.2|        7.0|                 N| 2025-09-21 08:15:00|
|        1|  trip1|          2.5|       10.0|                 N| 2025-09-20 10:00:00|
|        2|  trip2|          3.7|       15.5|                 Y| 2025-09-20 12:30:00|
+---------+-------+-------------+-----------+------------------+--------------------+



In [14]:
spark.sql("""
    SELECT * FROM hive_prod.demo.nyc_taxis
    WHERE trip_start_timestamp >= '2025-09-20 00:00:00'
    AND trip_start_timestamp < '2025-09-21 00:00:00'
""").show()

+---------+-------+-------------+-----------+------------------+--------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|trip_start_timestamp|
+---------+-------+-------------+-----------+------------------+--------------------+
|        1|  trip1|          2.5|       10.0|                 N| 2025-09-20 10:00:00|
|        2|  trip2|          3.7|       15.5|                 Y| 2025-09-20 12:30:00|
+---------+-------+-------------+-----------+------------------+--------------------+



In [15]:
# Insert new data (new snapshot)
new_data = [(3, "trip4", 4.0, 20.0, "N", "2025-09-21 09:00:00")]
new_df = spark.createDataFrame(new_data, ["vendor_id", "trip_id", "trip_distance", "fare_amount", "store_and_fwd_flag", "trip_start_timestamp"]) \
    .withColumn("trip_start_timestamp", to_timestamp("trip_start_timestamp"))
new_df.write.mode("append").saveAsTable("hive_prod.demo.nyc_taxis")

# Get snapshots
snapshots = spark.sql("SELECT * FROM hive_prod.demo.nyc_taxis.snapshots").collect()
snapshot_id = snapshots[0]["snapshot_id"]  # First snapshot

In [16]:
# Query first snapshot
spark.sql(f"SELECT * FROM hive_prod.demo.nyc_taxis VERSION AS OF {snapshot_id}").show()

+---------+-------+-------------+-----------+------------------+--------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|trip_start_timestamp|
+---------+-------+-------------+-----------+------------------+--------------------+
|        1|  trip3|          1.2|        7.0|                 N| 2025-09-21 08:15:00|
|        1|  trip1|          2.5|       10.0|                 N| 2025-09-20 10:00:00|
|        2|  trip2|          3.7|       15.5|                 Y| 2025-09-20 12:30:00|
+---------+-------+-------------+-----------+------------------+--------------------+



In [20]:
spark.sql("DESCRIBE hive_prod.demo.nyc_taxis").show(truncate=False)

+--------------------+--------------------------+-------+
|col_name            |data_type                 |comment|
+--------------------+--------------------------+-------+
|vendor_id           |bigint                    |NULL   |
|trip_id             |string                    |NULL   |
|trip_distance       |double                    |NULL   |
|fare_amount         |double                    |NULL   |
|store_and_fwd_flag  |string                    |NULL   |
|trip_start_timestamp|timestamp                 |NULL   |
|passenger_count     |int                       |NULL   |
|                    |                          |       |
|# Partitioning      |                          |       |
|Part 0              |days(trip_start_timestamp)|       |
+--------------------+--------------------------+-------+



In [21]:
spark.sql("DROP TABLE hive_prod.demo.nyc_taxis")

DataFrame[]

In [22]:
# Drop existing table (optional)
spark.sql("DROP TABLE IF EXISTS hive_prod.demo.nyc_taxis")

# Create table with passenger_count included
spark.sql("""
    CREATE TABLE IF NOT EXISTS hive_prod.demo.nyc_taxis (
        vendor_id BIGINT,
        trip_id STRING,
        trip_distance DOUBLE,
        fare_amount DOUBLE,
        store_and_fwd_flag STRING,
        trip_start_timestamp TIMESTAMP,
        passenger_count INT
    ) USING iceberg
    PARTITIONED BY (days(trip_start_timestamp))
    TBLPROPERTIES ('format-version' = '2')
    LOCATION 's3a://warehouse/demo/nyc_taxis'
""")

# Verify table
spark.sql("SHOW TABLES IN hive_prod.demo").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     demo|nyc_taxis|      false|
+---------+---------+-----------+



In [23]:
from pyspark.sql.functions import to_timestamp

# Create sample DataFrame
data = [
    (1, "trip1", 2.5, 10.0, "N", "2025-09-20 10:00:00", 1),
    (2, "trip2", 3.7, 15.5, "Y", "2025-09-20 12:30:00", 2),
    (1, "trip3", 1.2, 7.0, "N", "2025-09-21 08:15:00", 3)
]
df = spark.createDataFrame(data, ["vendor_id", "trip_id", "trip_distance", "fare_amount", "store_and_fwd_flag", "trip_start_timestamp", "passenger_count"]) \
    .withColumn("trip_start_timestamp", to_timestamp("trip_start_timestamp"))

# Write to Iceberg table
df.write.mode("append").saveAsTable("hive_prod.demo.nyc_taxis")

# Verify data
spark.sql("SELECT * FROM hive_prod.demo.nyc_taxis").show()

+---------+-------+-------------+-----------+------------------+--------------------+---------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|trip_start_timestamp|passenger_count|
+---------+-------+-------------+-----------+------------------+--------------------+---------------+
|        1|  trip3|          1.2|        7.0|                 N| 2025-09-21 08:15:00|              3|
|        1|  trip1|          2.5|       10.0|                 N| 2025-09-20 10:00:00|              1|
|        2|  trip2|          3.7|       15.5|                 Y| 2025-09-20 12:30:00|              2|
+---------+-------+-------------+-----------+------------------+--------------------+---------------+



In [24]:
spark.sql("""
    SELECT * FROM hive_prod.demo.nyc_taxis
    WHERE trip_start_timestamp >= '2025-09-20 00:00:00'
    AND trip_start_timestamp < '2025-09-21 00:00:00'
""").show()

+---------+-------+-------------+-----------+------------------+--------------------+---------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|trip_start_timestamp|passenger_count|
+---------+-------+-------------+-----------+------------------+--------------------+---------------+
|        1|  trip1|          2.5|       10.0|                 N| 2025-09-20 10:00:00|              1|
|        2|  trip2|          3.7|       15.5|                 Y| 2025-09-20 12:30:00|              2|
+---------+-------+-------------+-----------+------------------+--------------------+---------------+



In [25]:
# Insert new data
new_data = [(3, "trip4", 4.0, 20.0, "N", "2025-09-21 09:00:00", 4)]
new_df = spark.createDataFrame(new_data, ["vendor_id", "trip_id", "trip_distance", "fare_amount", "store_and_fwd_flag", "trip_start_timestamp", "passenger_count"]) \
    .withColumn("trip_start_timestamp", to_timestamp("trip_start_timestamp"))
new_df.write.mode("append").saveAsTable("hive_prod.demo.nyc_taxis")

# Get snapshots
snapshots = spark.sql("SELECT * FROM hive_prod.demo.nyc_taxis.snapshots").collect()
snapshot_id = snapshots[0]["snapshot_id"]  # First snapshot

# Query first snapshot
spark.sql(f"SELECT * FROM hive_prod.demo.nyc_taxis VERSION AS OF {snapshot_id}").show()

+---------+-------+-------------+-----------+------------------+--------------------+---------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|trip_start_timestamp|passenger_count|
+---------+-------+-------------+-----------+------------------+--------------------+---------------+
|        1|  trip3|          1.2|        7.0|                 N| 2025-09-21 08:15:00|              3|
|        1|  trip1|          2.5|       10.0|                 N| 2025-09-20 10:00:00|              1|
|        2|  trip2|          3.7|       15.5|                 Y| 2025-09-20 12:30:00|              2|
+---------+-------+-------------+-----------+------------------+--------------------+---------------+



In [26]:
# Add new column
spark.sql("""
    ALTER TABLE hive_prod.demo.nyc_taxis
    ADD COLUMNS (tip_amount DOUBLE)
""")

# Insert data with new column
new_data = [(4, "trip5", 5.0, 25.0, "N", "2025-09-21 10:00:00", 2, 3.0)]
new_df = spark.createDataFrame(new_data, ["vendor_id", "trip_id", "trip_distance", "fare_amount", "store_and_fwd_flag", "trip_start_timestamp", "passenger_count", "tip_amount"]) \
    .withColumn("trip_start_timestamp", to_timestamp("trip_start_timestamp"))
new_df.write.mode("append").saveAsTable("hive_prod.demo.nyc_taxis")

# Verify schema and data
spark.sql("DESCRIBE hive_prod.demo.nyc_taxis").show()
spark.sql("SELECT * FROM hive_prod.demo.nyc_taxis WHERE tip_amount IS NOT NULL").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|           vendor_id|              bigint|   NULL|
|             trip_id|              string|   NULL|
|       trip_distance|              double|   NULL|
|         fare_amount|              double|   NULL|
|  store_and_fwd_flag|              string|   NULL|
|trip_start_timestamp|           timestamp|   NULL|
|     passenger_count|                 int|   NULL|
|          tip_amount|              double|   NULL|
|                    |                    |       |
|      # Partitioning|                    |       |
|              Part 0|days(trip_start_t...|       |
+--------------------+--------------------+-------+

+---------+-------+-------------+-----------+------------------+--------------------+---------------+----------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|trip_start_timestamp|passenger_count|tip_a

In [27]:
# Delete a specific row
spark.sql("""
    DELETE FROM hive_prod.demo.nyc_taxis
    WHERE trip_id = 'trip2'
""")

# Verify deletion
spark.sql("SELECT * FROM hive_prod.demo.nyc_taxis WHERE trip_id = 'trip2'").show()

+---------+-------+-------------+-----------+------------------+--------------------+---------------+----------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|trip_start_timestamp|passenger_count|tip_amount|
+---------+-------+-------------+-----------+------------------+--------------------+---------------+----------+
+---------+-------+-------------+-----------+------------------+--------------------+---------------+----------+



In [28]:
# Compact small files
spark.sql("CALL hive_prod.system.rewrite_data_files('demo.nyc_taxis')").show()

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|                         0|                     0|                    0|                      0|
+--------------------------+----------------------+---------------------+-----------------------+



In [29]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|     demo|
+---------+



In [30]:
spark.sql("SHOW TABLES IN hive_prod.demo").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     demo|nyc_taxis|      false|
+---------+---------+-----------+



In [31]:
spark.sql("SHOW TABLES IN default").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [32]:
spark.sql("DESCRIBE hive_prod.demo.nyc_taxis").show(truncate=False)

+--------------------+--------------------------+-------+
|col_name            |data_type                 |comment|
+--------------------+--------------------------+-------+
|vendor_id           |bigint                    |NULL   |
|trip_id             |string                    |NULL   |
|trip_distance       |double                    |NULL   |
|fare_amount         |double                    |NULL   |
|store_and_fwd_flag  |string                    |NULL   |
|trip_start_timestamp|timestamp                 |NULL   |
|passenger_count     |int                       |NULL   |
|tip_amount          |double                    |NULL   |
|                    |                          |       |
|# Partitioning      |                          |       |
|Part 0              |days(trip_start_timestamp)|       |
+--------------------+--------------------------+-------+

